# Preprocessor — manual functional test

Hands-on check of everything in `src/core_engine/pipeline/preprocessor.py`:

1. `probe_video` on a real clip (metadata)
2. `VideoValidationError` paths (missing / corrupt / audio-only)
3. `resolve_output_geometry` (fps + scaling math)
4. `VideoPreprocessor` properties + generated ffmpeg command
5. `extract_frames_generator` (shape, dtype, fps downsampling)
6. spatial downscaling (`max_dimension`)
7. `extract_all_frames` vs generator
8. `max_frames` cap
9. look at a real frame

Run: **Kernel → Restart & Run All**. Requires `ffmpeg`/`ffprobe` on PATH.
Uses `tests/samples/sample1.mp4` (or `packages/core/samples/`), else generates a synthetic clip.

In [6]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Make `core_engine` importable no matter the notebook cwd
try:
    import core_engine
except ModuleNotFoundError:
    print("core_engine not found, trying to add it to sys.path...")
    here = Path.cwd()
    for cand in [
        here / "src",                    # cwd == packages/core
        here.parent / "src",             # cwd == packages/core/tests
        here / "packages" / "core" / "src",  # cwd == repo root
    ]:
        if (cand / "core_engine" / "__init__.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            break
    import core_engine

import numpy as np
from core_engine.config import VectorizeConfig
from core_engine.pipeline.preprocessor import (
    VideoMetadata,
    VideoPreprocessor,
    VideoValidationError,
    probe_video,
    resolve_output_geometry,
)

assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "ffmpeg/ffprobe missing from PATH"
print("core_engine from:", core_engine.__file__)

core_engine from: /Users/mac/Desktop/work/VectoRise/packages/core/src/core_engine/__init__.py


In [3]:
def find_sample() -> Path | None:
    here = Path.cwd()
    candidates = [
        here / "samples" / "sample1.mp4",                  # cwd == tests/
        here / "tests" / "samples" / "sample1.mp4",        # cwd == packages/core
        here.parent / "samples" / "sample1.mp4",           # cwd == packages/core/tests
        here / "packages" / "core" / "samples" / "sample1.mp4",  # cwd == repo root
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return None


def make_synthetic(path: Path, w=64, h=48, fps=30, duration=1) -> Path:
    subprocess.run(
        ["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i",
         f"testsrc=size={w}x{h}:rate={fps}:duration={duration}",
         "-pix_fmt", "yuv420p", "-c:v", "libx264", str(path)],
        check=True,
    )
    return path


CLIP = find_sample()
if CLIP is None:
    _tmp = Path(tempfile.mkdtemp(prefix="vecto_"))
    CLIP = make_synthetic(_tmp / "synthetic.mp4")
    print("No sample found — generated synthetic:", CLIP)
else:
    print(f"Using sample clip: {CLIP} ({CLIP.stat().st_size / 1e6:.2f} MB)")

Using sample clip: /Users/mac/Desktop/work/VectoRise/packages/core/tests/samples/sample1.mp4 (1.21 MB)


## 1. Probe the clip — `probe_video`

In [4]:
meta = probe_video(CLIP)
print(f"path       : {meta.path}")
print(f"resolution : {meta.width}x{meta.height}")
print(f"fps        : {meta.fps}")
print(f"duration   : {meta.duration:.3f}s" if meta.duration else "duration   : unknown")
print(f"frames     : {meta.frame_count}")
print(f"pix_fmt    : {meta.pix_fmt}   codec: {meta.codec}")
assert meta.width > 0 and meta.height > 0 and meta.fps > 0

path       : /Users/mac/Desktop/work/VectoRise/packages/core/tests/samples/sample1.mp4
resolution : 1280x720
fps        : 24.0
duration   : 25.417s
frames     : 610
pix_fmt    : yuv420p   codec: h264


## 2. Error paths — `VideoValidationError`

In [8]:
def expect_validation_error(label, fn):
    try:
        fn()
    except VideoValidationError as e:
        print(f"OK  {label}: VideoValidationError: {e}")
    else:
        raise AssertionError(f"{label} did NOT raise VideoValidationError")


expect_validation_error("missing file", lambda: probe_video(CLIP.parent / "does-not-exist.mp4"))

_bad = Path(tempfile.mkdtemp(prefix="vecto_")) / "bad.mp4"
_bad.write_bytes(b"not a video" * 100)
expect_validation_error("corrupt file", lambda: probe_video(_bad))

_audio = Path(tempfile.mkdtemp(prefix="vecto_")) / "audio.m4a"
subprocess.run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i",
                "anullsrc=r=44100:cl=mono:d=1", "-c:a", "aac", str(_audio)], check=True)
expect_validation_error("audio-only file", lambda: probe_video(_audio))

OK  missing file: VideoValidationError: Video file not found: /Users/mac/Desktop/work/VectoRise/packages/core/tests/samples/does-not-exist.mp4
OK  corrupt file: VideoValidationError: ffprobe failed for /var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/vecto_p94fcd3q/bad.mp4: [mov,mp4,m4a,3gp,3g2,mj2 @ 0x8cac18000] moov atom not found
/var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/vecto_p94fcd3q/bad.mp4: Invalid data found when processing input
OK  audio-only file: VideoValidationError: No readable video stream in: /var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/vecto_yc9iwxam/audio.m4a


## 3. Output geometry math — `resolve_output_geometry`

In [ ]:
base = VectorizeConfig(input_path=str(CLIP), output_path="")
for cfg in [
    base,
    VectorizeConfig(str(CLIP), "", target_fps=10.0),
    VectorizeConfig(str(CLIP), "", max_dimension=320),
    VectorizeConfig(str(CLIP), "", target_fps=12.0, max_dimension=720),
]:
    w, h, fps = resolve_output_geometry(meta, cfg)
    print(f"target_fps={cfg.target_fps}  max_dim={cfg.max_dimension}  ->  {w}x{h} @ {fps}fps")
    assert w % 2 == 0 and h % 2 == 0, "dims must stay even"

## 4. Preprocessor object — properties + ffmpeg command

In [ ]:
cfg = VectorizeConfig(input_path=str(CLIP), output_path="", target_fps=10.0, max_dimension=720)
pre = VideoPreprocessor(str(CLIP), cfg)
print("metadata :", pre.metadata)
print("info == metadata:", pre.info == pre.metadata)
print(f"output   : {pre.output_width}x{pre.output_height} @ {pre.output_fps}fps")
print("estimated:", pre.estimated_frames, "frames")
print("filters  :", pre._video_filters())
print("ffmpeg   :", " ".join(pre._ffmpeg_cmd()))

## 5. Streaming decode — `extract_frames_generator`

In [ ]:
frames = []
for f in pre.extract_frames_generator():
    assert isinstance(f, np.ndarray) and f.dtype == np.uint8
    assert f.shape == (pre.output_height, pre.output_width, 3), f.shape
    frames.append(f)
print(f"decoded {len(frames)} frames, each {frames[0].shape} {frames[0].dtype}")
if meta.duration:
    print(f"expected ~{meta.duration * pre.output_fps:.0f} (duration x out_fps)")
f0 = frames[0]
print(f"frame0 min/max/mean: {f0.min()} / {f0.max()} / {f0.mean():.1f}")

## 6. Spatial downscaling — `max_dimension`

In [ ]:
small = VideoPreprocessor(str(CLIP), VectorizeConfig(str(CLIP), "", max_dimension=160))
print("downscaled to:", small.output_width, "x", small.output_height)
assert max(small.output_width, small.output_height) <= 160
got = [f for _, f in zip(range(3), small.extract_frames_generator())]
assert all(f.shape == (small.output_height, small.output_width, 3) for f in got)
print("first 3 downscaled frames OK:", got[0].shape, got[0].dtype)

## 7. Batch loader — `extract_all_frames` == generator

In [ ]:
batch_cfg = VectorizeConfig(str(CLIP), "", target_fps=10.0, max_dimension=160)
bpre = VideoPreprocessor(str(CLIP), batch_cfg)
batch = bpre.extract_all_frames()
gen = list(bpre.extract_frames_generator())
assert len(batch) == len(gen) and all(np.array_equal(a, b) for a, b in zip(batch, gen))
print(f"batch OK: {len(batch)} frames, identical to generator output")

## 8. Frame cap — `max_frames`

In [ ]:
capped = VideoPreprocessor(str(CLIP), VectorizeConfig(str(CLIP), "", target_fps=10.0, max_frames=5))
five = capped.extract_all_frames()
assert len(five) == 5
print("max_frames=5 ->", len(five), "frames")

## 9. Look at a real frame

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed — skipping display")
    print("frame0 channel means (R,G,B):", frames[0].mean(axis=(0, 1)).round(1))
else:
    plt.figure(figsize=(8, 4.5))
    plt.imshow(frames[0])
    plt.axis("off")
    plt.title(f"frame 0 — {pre.output_width}x{pre.output_height} RGB")
    plt.show()

All green — every `preprocessor.py` functionality verified ✔